# Phase 5: Walk-Forward Validation, Regime-Conditional cvxpy Optimization & Honest Backtest

**This is the submission-grade pipeline.** Phases 3-4 were a useful first pass, but they have
two gaps relative to the problem statement that this notebook fixes:

1. **Lookahead bias in the HMM.** Phase 3 fit the HMM once on the *entire* 2015-2024 history and
   then labeled every day, including 2016, with a model that had already seen 2020 and 2022.
   This is the exact failure mode the Project Guide notebook calls out by name in Section 7:
   *"Fitting the HMM once on the entire dataset, then 'predicting' regimes for early dates — the
   model has seen the whole future distribution of states before labeling day 1."* Every backtest
   number built on top of that (all of Phase 4) inherits the leak.
2. **No actual convex optimization.** Phase 4 hand-picked fixed percentages per regime. The
   tech stack and roadmap explicitly call for solving regime-specific objectives with `cvxpy`.
   That never happened.

This notebook builds a proper **walk-forward harness**: the HMM is refit from scratch inside
every training fold, all feature normalization uses training-fold statistics only, and a
**`cvxpy` quadratic program is solved per regime, per fold**, using only that fold's training
data. The result is a continuous **out-of-sample** equity curve — the kind of result that
should hold up if the graders re-run this on data the model never saw.

**How this fits with Phases 3-4:** keep them in the repo. They're a legitimate, useful
demonstration of *why* walk-forward matters (mirrors the LEAKY-vs-SAFE z-score comparison you
already did in Phase 2) — just be explicit in your README that Phase 3/4 are the in-sample
exploratory pass and Phase 5 is the actual out-of-sample result you're standing behind.


## 1. Environment & Config

Everything that's a modeling choice lives in one place below, so you can tune it and defend
each choice in your README (`why 3 regimes`, `why these risk-aversion levels`, etc.) without
hunting through the notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cvxpy as cp
from hmmlearn import hmm
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ---------------- Config: every modeling choice in one place ----------------
ASSET_COLS      = ["NSE", "IEF", "GLD"]     # <- change if your Phase 1 CSV used different tickers
VIX_COL         = "VIX"
N_STATES        = 3                          # Bull / Bear / Crisis
MIN_TRAIN       = 750                        # ~3 trading years before the first OOS prediction
TEST_SIZE       = 63                         # refit the HMM every ~1 trading quarter
COST_BPS        = 7.5                        # midpoint of the 5-10 bps range required by the brief
MIN_REGIME_OBS  = 20                         # min. training days in a regime before trusting its own mean/cov
RISK_AVERSION   = {"Bull": 2.0, "Bear": 6.0, "Crisis": 25.0}   # higher = more conservative objective
MAX_WEIGHT      = 0.65                       # per-asset cap so the QP can't dump 100% into one asset
VOL_FEATURE     = "vol_21d"                  # feature used to tell HMM states apart (highest vol = Crisis)
MIN_DWELL       = 5                          # a new raw regime must persist this many days before we ACT on it

print("Config loaded.")


## 2. Load Phase 1 & Phase 2 outputs

**Important:** this loads `features_raw.csv`, not `features_normalized.csv`. Phase 2's
normalized file used one continuous *expanding* z-score across the whole timeline. That's
causally valid on its own, but it isn't the same thing as walk-forward validation, where the
brief specifically wants training-fold statistics recomputed fresh every fold (Project Guide,
Section 8: *"Compute normalization stats ONLY on train_data... test uses TRAIN's mu/sigma,
never its own"*). So we z-score inside the loop below, per fold, from raw features.

If you don't have these two files yet, run Phase 1 and Phase 2 first. As a safety net (so this
notebook is runnable standalone for a sanity check), the next cell falls back to a small
synthetic dataset if the real files aren't found — any output built on synthetic data is
labeled clearly so you never mistake it for a real result.


In [ ]:
import os

DATA_DIR = "data"
USING_SYNTHETIC = False

def _make_synthetic_demo_data():
    """Small synthetic dataset with an injected regime structure, used ONLY if the real
    Phase 1/2 output files are missing, so this notebook can still be sanity-checked end to end."""
    dates = pd.date_range("2015-01-01", "2024-01-01", freq="B")
    n = len(dates)
    true_regime = np.zeros(n, dtype=int)
    def stamp(s, e, v):
        true_regime[dates.searchsorted(pd.Timestamp(s)):dates.searchsorted(pd.Timestamp(e))] = v
    stamp("2015-08-01", "2015-10-15", 1)
    stamp("2018-10-01", "2018-12-24", 1)
    stamp("2020-02-20", "2020-04-15", 2)
    stamp("2020-04-15", "2020-06-30", 1)
    stamp("2022-01-01", "2022-06-30", 1)
    stamp("2022-06-01", "2022-06-30", 2)
    params = {0: (0.0006, 0.008, 0.0001, 0.003, 0.0001, 0.006),
              1: (-0.0004, 0.014, 0.0002, 0.004, 0.0002, 0.008),
              2: (-0.0020, 0.032, 0.0004, 0.006, 0.0006, 0.014)}
    nse, ief, gld, vix, vlvl = np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n), 15.0
    for i in range(n):
        mu_n, s_n, mu_i, s_i, mu_g, s_g = params[true_regime[i]]
        z = np.random.normal()
        nse[i] = mu_n + s_n * z
        ief[i] = mu_i + s_i * np.random.normal() - 0.15 * s_i * z
        gld[i] = mu_g + s_g * np.random.normal() - 0.25 * s_g * z
        target = {0: 14, 1: 22, 2: 38}[true_regime[i]]
        vlvl = max(9, vlvl + 0.15 * (target - vlvl) + np.random.normal(0, 1.2))
        vix[i] = vlvl
    mret = pd.DataFrame({"NSE": nse, "IEF": ief, "GLD": gld, "VIX": vix}, index=dates)
    close = 100 * np.exp(mret["NSE"].cumsum())
    feat = pd.DataFrame(index=dates)
    for w in [5, 21, 63, 126]:
        feat[f"mom_{w}d"] = close.pct_change(w)
    for w in [5, 21, 63]:
        feat[f"vol_{w}d"] = mret["NSE"].rolling(w).std() * np.sqrt(252)
    feat["avg_cross_vol"] = (mret[["NSE", "IEF", "GLD"]].rolling(21).std() * np.sqrt(252)).mean(axis=1)
    feat["roll_corr_nse_gld"] = mret["NSE"].rolling(63).corr(mret["GLD"])
    feat["vix_level"] = mret["VIX"]
    feat["vix_ma_21"] = mret["VIX"].rolling(21).mean()
    feat["vix_roc_21"] = mret["VIX"].pct_change(21)
    return mret, feat.dropna()

try:
    master = pd.read_csv(f"{DATA_DIR}/master_returns.csv", index_col=0, parse_dates=True)
    feat_raw = pd.read_csv(f"{DATA_DIR}/features_raw.csv", index_col=0, parse_dates=True)
    print("Loaded real Phase 1 / Phase 2 output from data/.")
except FileNotFoundError:
    print("!! data/master_returns.csv or data/features_raw.csv not found.")
    print("!! Falling back to a SYNTHETIC demo dataset so this notebook still runs end-to-end.")
    print("!! Re-run this after Phase 1 & 2 to get REAL results -- everything below will say SYNTHETIC.")
    USING_SYNTHETIC = True
    master, feat_raw = _make_synthetic_demo_data()

missing_assets = [c for c in ASSET_COLS if c not in master.columns]
if missing_assets:
    print(f"!! ASSET_COLS {missing_assets} not found in master_returns.csv columns: {list(master.columns)}")
    print("!! Update ASSET_COLS in the config cell above to match your actual column names.")

common_idx = master.index.intersection(feat_raw.index)
master, feat_raw = master.loc[common_idx], feat_raw.loc[common_idx]
asset_ret = master[ASSET_COLS]
n_assets = len(ASSET_COLS)

print(f"\nAligned dataset: {feat_raw.shape[0]} rows, {feat_raw.shape[1]} features, assets = {ASSET_COLS}")
print(f"Date range: {feat_raw.index.min().date()} to {feat_raw.index.max().date()}")
feat_raw.head()


## 3. Walk-forward split function

Contiguous, non-overlapping folds: each day is predicted exactly once, by a model trained
only on days strictly before it. `TEST_SIZE = 63` means the HMM and the regime-conditional
optimizer are both **refit from scratch every ~quarter**, which is what "walk-forward" means
per the Project Guide's Section 8 diagram.


In [ ]:
def walk_forward_splits(n_obs, min_train, test_size):
    splits, start_test = [], min_train
    while start_test < n_obs:
        end_test = min(start_test + test_size, n_obs)
        splits.append((np.arange(0, start_test), np.arange(start_test, end_test)))
        start_test = end_test
    return splits

splits = walk_forward_splits(len(feat_raw), MIN_TRAIN, TEST_SIZE)
print(f"{len(splits)} walk-forward folds")
for i, (tr, te) in enumerate(splits[:3] + splits[-2:]):
    tag = "..." if i == 3 and len(splits) > 5 else ""
    print(f"Fold {i}: train[0:{tr[-1]}] ({feat_raw.index[tr[0]].date()} to {feat_raw.index[tr[-1]].date()})  "
          f"test[{te[0]}:{te[-1]}] ({feat_raw.index[te[0]].date()} to {feat_raw.index[te[-1]].date()})")


## 4. Per-fold HMM: refit, predict, and label states

Inside each fold:
1. Compute `mu`/`sigma` from the **training slice only**, z-score both train and test with it.
2. Fit a fresh `GaussianHMM(n_components=3)` on the training z-scores only.
3. Call `.predict()` (Viterbi) on the test z-scores — this uses the already-fitted transition
   and emission parameters, it does **not** refit, so there's no leakage in this step either.
4. Since HMM state indices (0, 1, 2) are arbitrary, label them by each state's mean on the
   volatility feature: highest → Crisis, lowest → Bull, middle → Bear (same heuristic the
   Project Guide uses in Section 9.1).


In [ ]:
def label_states_by_vol(model, vol_col_idx):
    means = model.means_[:, vol_col_idx]
    order = np.argsort(means)   # ascending volatility
    return {order[0]: "Bull", order[1]: "Bear", order[2]: "Crisis"}

vol_col_idx = feat_raw.columns.get_loc(VOL_FEATURE)
print(f"Using '{VOL_FEATURE}' (column {vol_col_idx}) to tell states apart.")


## 5. Causal regime-dwell filter

Raw day-by-day Viterbi output tends to flip far more often than a real macro "regime" should
— in testing this pipeline, raw labels averaged only ~6 days before switching, which doesn't
match the idea of a persistent Bull/Bear/Crisis state, and it also manufactures unrealistic
transaction costs (Phase 4's rebalancing-cost checklist item is meant to catch exactly this).

The fix here is a **causal hysteresis filter**: the *executed* regime only switches once a new
candidate label has appeared for `MIN_DWELL` consecutive raw days. Each day's smoothed decision
depends only on that day and earlier days — nothing forward-looking — so this doesn't reopen
the lookahead-bias problem, it just stops the strategy from chasing single-day noise.


In [ ]:
def apply_min_dwell(labels, min_dwell):
    """labels: array-like of raw per-day regime strings, already in chronological order.
    Returns the EXECUTED regime for each day. Strictly causal (backward-looking only)."""
    labels = pd.Series(labels).reset_index(drop=True)
    active = labels.iloc[0]
    candidate, candidate_count = None, 0
    out = []
    for lbl in labels:
        if lbl == active:
            candidate, candidate_count = None, 0
        else:
            if lbl == candidate:
                candidate_count += 1
            else:
                candidate, candidate_count = lbl, 1
            if candidate_count >= min_dwell:
                active, candidate, candidate_count = candidate, None, 0
        out.append(active)
    return out


## 6. Regime-conditional statistics + `cvxpy` optimization per fold

This is the piece Phase 4 skipped. For each fold, using **training-period asset returns only**:

- Split training days by their (raw, training-only) regime label and compute each regime's
  own annualized mean vector and covariance matrix.
- If a regime has fewer than `MIN_REGIME_OBS` training days (common early on, before a Crisis
  has actually occurred yet in the training window), fall back to the full training-period
  mean/covariance rather than fitting on a handful of noisy points.
- Solve a genuinely different **convex objective per regime**, per the roadmap:
  - **Bull** — mean-variance with low risk aversion (`maximize μᵀw − λ·wᵀΣw`, small λ): growth-tilted.
  - **Bear** — the same objective with a higher λ: more defensive, still return-seeking.
  - **Crisis** — pure minimum-variance (`minimize wᵀΣw`): capital preservation, ignore expected
    return entirely, matching the roadmap's "minimize volatility in Crisis."
- Constraints: fully invested, long-only, and a `MAX_WEIGHT` cap per asset so the QP can't
  collapse into a single-asset corner solution (a real risk with only 3 assets and no other
  constraints — worth mentioning in your README as a deliberate design choice).


In [ ]:
def solve_regime_weights(r_mu, r_cov, regime):
    r_cov = r_cov + np.eye(n_assets) * 1e-6   # numerical stability
    w = cp.Variable(n_assets)
    if regime == "Crisis":
        objective = cp.Minimize(cp.quad_form(w, cp.psd_wrap(r_cov)))
    else:
        lam = RISK_AVERSION[regime]
        objective = cp.Maximize(r_mu @ w - lam * cp.quad_form(w, cp.psd_wrap(r_cov)))
    constraints = [cp.sum(w) == 1, w >= 0, w <= MAX_WEIGHT]
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.OSQP)
    if w.value is None:
        return np.ones(n_assets) / n_assets   # fallback if the solver fails on a given fold
    w_sol = np.clip(w.value, 0, None)
    return w_sol / w_sol.sum()


## 7. The walk-forward loop

Puts it all together: for every fold, refit the HMM, label its states, solve the three
regime portfolios from training data, and record the raw test-period regime labels. Nothing
here uses information from outside its own fold's training window.


In [ ]:
oos_regime_raw = pd.Series(index=feat_raw.index, dtype=object)
fold_weights_by_day = pd.Series(index=feat_raw.index, dtype=object)   # which fold applies to each OOS day
all_fold_weights = []
fold_transmats, fold_label_maps = [], []

for fold_i, (train_idx, test_idx) in enumerate(splits):
    train_feat, test_feat = feat_raw.iloc[train_idx], feat_raw.iloc[test_idx]

    # --- normalize using TRAIN stats only ---
    mu, sigma = train_feat.mean(), train_feat.std().replace(0, 1e-8)
    train_z, test_z = (train_feat - mu) / sigma, (test_feat - mu) / sigma

    # --- fit HMM fresh on this fold's training data only ---
    model = hmm.GaussianHMM(n_components=N_STATES, covariance_type="diag", n_iter=200, random_state=42)
    model.fit(train_z.values)
    train_states = model.predict(train_z.values)
    test_states = model.predict(test_z.values)   # Viterbi on frozen params, no refit

    label_map = label_states_by_vol(model, vol_col_idx)
    fold_label_maps.append(label_map)
    fold_transmats.append(model.transmat_)

    train_labels = pd.Series(train_states, index=train_feat.index).map(label_map)
    test_labels = pd.Series(test_states, index=test_feat.index).map(label_map)
    oos_regime_raw.loc[test_feat.index] = test_labels.values
    fold_weights_by_day.loc[test_feat.index] = fold_i

    # --- regime-conditional stats from TRAIN returns only, then solve cvxpy per regime ---
    train_ret = asset_ret.loc[train_feat.index]
    full_mu, full_cov = train_ret.mean().values * 252, train_ret.cov().values * 252

    fold_weights = {}
    for regime in ["Bull", "Bear", "Crisis"]:
        mask = (train_labels == regime).values
        if mask.sum() >= MIN_REGIME_OBS:
            r_mu = train_ret.loc[mask].mean().values * 252
            r_cov = train_ret.loc[mask].cov().values * 252
        else:
            r_mu, r_cov = full_mu, full_cov
        fold_weights[regime] = solve_regime_weights(r_mu, r_cov, regime)
    all_fold_weights.append(fold_weights)

oos_regime_raw = oos_regime_raw.dropna()

# --- causal dwell-time smoothing on the concatenated, time-ordered raw OOS labels ---
oos_regime = pd.Series(apply_min_dwell(oos_regime_raw, MIN_DWELL), index=oos_regime_raw.index)

# --- map each OOS day to the weight vector its own fold solved for its (smoothed) regime ---
daily_target_weight = pd.DataFrame(index=oos_regime.index, columns=ASSET_COLS, dtype=float)
for d in oos_regime.index:
    fold_i = fold_weights_by_day.loc[d]
    daily_target_weight.loc[d] = all_fold_weights[fold_i][oos_regime.loc[d]]

oos_ret = asset_ret.loc[daily_target_weight.index]

switches_raw = (oos_regime_raw != oos_regime_raw.shift()).sum()
switches_smoothed = (oos_regime != oos_regime.shift()).sum()
tag = " [SYNTHETIC DEMO DATA]" if USING_SYNTHETIC else ""
print(f"OOS period{tag}: {oos_ret.index.min().date()} to {oos_ret.index.max().date()}  ({len(oos_ret)} days)")
print(f"Raw regime switches: {switches_raw} (avg run {len(oos_regime)/switches_raw:.1f}d)  ->  "
      f"after dwell filter: {switches_smoothed} (avg run {len(oos_regime)/switches_smoothed:.1f}d)")
print("\nExecuted (out-of-sample) regime day-counts:")
print(oos_regime.value_counts())


## 8. Regime overlay on price (out-of-sample, honest this time)

Unlike Phase 3's version of this chart, every color band here reflects a label assigned by a
model that had **not yet seen** the days it's coloring.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
price_display = 100 * np.exp(asset_ret[ASSET_COLS[0]].cumsum())
ax.plot(price_display.index, price_display, color="black", lw=1, zorder=3)

colors = {"Bull": "#2ecc71", "Bear": "#e67e22", "Crisis": "#e74c3c"}
vals, idx = oos_regime.values, oos_regime.index
start = 0
for i in range(1, len(vals) + 1):
    if i == len(vals) or vals[i] != vals[start]:
        ax.axvspan(idx[start], idx[min(i, len(vals) - 1)], color=colors[vals[start]], alpha=0.25, lw=0)
        start = i

ax.set_xlim(oos_regime.index.min(), oos_regime.index.max())
handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.5) for c in colors.values()]
ax.legend(handles, colors.keys(), loc="upper left")
title = f"Out-of-Sample HMM Regimes on {ASSET_COLS[0]}"
if USING_SYNTHETIC:
    title += "  [SYNTHETIC DEMO DATA]"
ax.set_title(title)
plt.tight_layout()
plt.show()


## 9. Transition matrix (averaged across folds)

Each fold's raw transition matrix has states in an arbitrary order, so we reorder every fold's
matrix to a common Bull/Bear/Crisis axis (using that fold's own label map) before averaging.
The diagonal is what "sticky regimes" means in practice.


In [ ]:
names = ["Bull", "Bear", "Crisis"]
avg_transmat = np.zeros((3, 3))
for tm, lm in zip(fold_transmats, fold_label_maps):
    inv = {v: k for k, v in lm.items()}
    reordered = np.array([[tm[inv[a], inv[b]] for b in names] for a in names])
    avg_transmat += reordered
avg_transmat /= len(fold_transmats)

transition_df = pd.DataFrame(avg_transmat, index=names, columns=names)
print("Average transition matrix across all folds (rows = from, cols = to):")
print(transition_df.round(3))

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(avg_transmat, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(names)
ax.set_yticks(range(3)); ax.set_yticklabels(names)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{avg_transmat[i,j]:.2f}", ha="center", va="center",
                 color="white" if avg_transmat[i, j] > 0.5 else "black")
ax.set_title("Avg. Transition Matrix (fold-averaged)")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()


## 10. Backtest: apply weights out-of-sample, with transaction costs

Two backtest engines:

- **Regime-aware dynamic strategy** — buy-and-hold within a regime; a trade (and its cost)
  only happens when the *executed*, dwell-filtered target weight actually changes (a genuine
  regime switch, or a fold boundary that re-optimized the same regime slightly differently).
  This is what ties the transaction-cost line item to actual regime-flip frequency rather than
  to daily rebalancing noise.
- **Static benchmarks** (60/40, equal-weight) — rebalanced monthly, same `COST_BPS`, over the
  identical OOS window so the comparison is apples-to-apples.

Both gross (no cost) and net (with cost) returns are kept, per the checklist requirement to
show results "both with and without" transaction costs.


In [ ]:
def backtest_regime_aware(weights_df, returns_df, cost_bps):
    dates = returns_df.index
    held_w = weights_df.iloc[0].values.astype(float).copy()
    prev_target = held_w.copy()
    gross_rets, net_rets, turnovers = [], [], []
    for d in dates:
        target_w = weights_df.loc[d].values.astype(float)
        if not np.allclose(target_w, prev_target):
            turnover = 0.5 * np.abs(target_w - held_w).sum()
            held_w = target_w.copy()
        else:
            turnover = 0.0
        cost = turnover * (cost_bps / 10000.0)
        r = returns_df.loc[d].values
        g_ret = float(np.dot(held_w, r))
        net_rets.append(g_ret - cost)
        gross_rets.append(g_ret)
        turnovers.append(turnover)
        held_w = held_w * (1 + r)
        held_w = held_w / held_w.sum()
        prev_target = target_w
    return pd.DataFrame({"gross_ret": gross_rets, "net_ret": net_rets, "turnover": turnovers}, index=dates)

def static_benchmark(returns_df, weights, cost_bps):
    dates = returns_df.index
    target = np.array(weights)
    month_marker = dates.to_period("M")
    held_w = target.copy()
    gross_rets, net_rets, turnovers = [], [], []
    prev_period = None
    for d, period in zip(dates, month_marker):
        r = returns_df.loc[d].values
        if period != prev_period:
            turnover = 0.5 * np.abs(target - held_w).sum()
            held_w = target.copy()
        else:
            turnover = 0.0
        cost = turnover * (cost_bps / 10000.0)
        g_ret = float(np.dot(held_w, r))
        gross_rets.append(g_ret); net_rets.append(g_ret - cost); turnovers.append(turnover)
        held_w = held_w * (1 + r); held_w = held_w / held_w.sum()
        prev_period = period
    return pd.DataFrame({"gross_ret": gross_rets, "net_ret": net_rets, "turnover": turnovers}, index=dates)

dynamic_bt = backtest_regime_aware(daily_target_weight, oos_ret, COST_BPS)

# 60/40 defined as 60% first asset / 40% second asset / 0% third (adjust indices if your
# ASSET_COLS ordering differs from [equity, bonds, gold])
w_6040 = [0.60, 0.40, 0.0][:n_assets]
w_eq = [1 / n_assets] * n_assets
bench_6040 = static_benchmark(oos_ret, w_6040, COST_BPS)
bench_eq = static_benchmark(oos_ret, w_eq, COST_BPS)

print("Backtests complete.")


## 11. Performance summary

Sharpe, Sortino, Calmar, max drawdown, CAGR, annualized vol, and annualized turnover — the
full metric set the submission checklist asks for, computed for the dynamic strategy both
gross and net of cost, plus both static benchmarks.


In [ ]:
def perf_metrics(ret, freq=252):
    ret = ret.dropna()
    cagr = (1 + ret).prod() ** (freq / len(ret)) - 1
    vol = ret.std() * np.sqrt(freq)
    sharpe = (ret.mean() * freq) / (ret.std() * np.sqrt(freq) + 1e-12)
    downside = ret[ret < 0]
    sortino = (ret.mean() * freq) / (downside.std() * np.sqrt(freq) + 1e-12)
    equity = (1 + ret).cumprod()
    dd = equity / equity.cummax() - 1
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan
    return dict(CAGR=cagr, AnnVol=vol, Sharpe=sharpe, Sortino=sortino, MaxDD=max_dd, Calmar=calmar)

summary = pd.DataFrame({
    "Dynamic (gross)": perf_metrics(dynamic_bt["gross_ret"]),
    "Dynamic (net)":   perf_metrics(dynamic_bt["net_ret"]),
    "60/40 (net)":     perf_metrics(bench_6040["net_ret"]),
    "Equal-weight (net)": perf_metrics(bench_eq["net_ret"]),
}).T
summary["AnnTurnover"] = [
    dynamic_bt["turnover"].sum() / (len(dynamic_bt) / 252),
    dynamic_bt["turnover"].sum() / (len(dynamic_bt) / 252),
    bench_6040["turnover"].sum() / (len(bench_6040) / 252),
    bench_eq["turnover"].sum() / (len(bench_eq) / 252),
]

pd.set_option("display.float_format", lambda x: f"{x:0.4f}")
if USING_SYNTHETIC:
    print("!! SYNTHETIC DEMO DATA -- re-run on your real Phase 1/2 output for real numbers !!\n")
summary


## 12. Equity curves & drawdowns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

for label, ret, style in [("Dynamic (net)", dynamic_bt["net_ret"], "-"),
                           ("Dynamic (gross)", dynamic_bt["gross_ret"], "--"),
                           ("60/40 (net)", bench_6040["net_ret"], "-"),
                           ("Equal-weight (net)", bench_eq["net_ret"], "-")]:
    eq = (1 + ret).cumprod()
    axes[0].plot(eq.index, eq, style, label=label, lw=1.4 if "Dynamic" in label else 1.0)
axes[0].set_yscale("log")
axes[0].legend(loc="upper left")
title = "Out-of-Sample Equity Curves"
if USING_SYNTHETIC:
    title += "  [SYNTHETIC DEMO DATA]"
axes[0].set_title(title)

for label, ret in [("Dynamic (net)", dynamic_bt["net_ret"]), ("60/40 (net)", bench_6040["net_ret"]),
                    ("Equal-weight (net)", bench_eq["net_ret"])]:
    eq = (1 + ret).cumprod()
    dd = eq / eq.cummax() - 1
    axes[1].plot(dd.index, dd, label=label)
axes[1].set_title("Drawdown")
axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()


## 13. Save outputs for the README / submission

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
oos_regime.to_csv(f"{DATA_DIR}/oos_regimes_walkforward.csv", header=["regime"])
daily_target_weight.to_csv(f"{DATA_DIR}/oos_target_weights.csv")
dynamic_bt.to_csv(f"{DATA_DIR}/oos_dynamic_returns.csv")
bench_6040.to_csv(f"{DATA_DIR}/oos_bench_6040_returns.csv")
bench_eq.to_csv(f"{DATA_DIR}/oos_bench_equalweight_returns.csv")
summary.to_csv(f"{DATA_DIR}/oos_performance_summary.csv")
transition_df.to_csv(f"{DATA_DIR}/oos_transition_matrix_avg.csv")

print("Saved to data/:")
for f in ["oos_regimes_walkforward.csv", "oos_target_weights.csv", "oos_dynamic_returns.csv",
          "oos_bench_6040_returns.csv", "oos_bench_equalweight_returns.csv",
          "oos_performance_summary.csv", "oos_transition_matrix_avg.csv"]:
    print(" -", f)
if USING_SYNTHETIC:
    print("\n!! These files were built on SYNTHETIC demo data. Re-run this notebook after")
    print("!! confirming data/master_returns.csv and data/features_raw.csv exist, then re-save.")


## 14. Notes for your README

Things worth writing up explicitly, since the README is a graded deliverable:

- **Why walk-forward changes the story.** If you compare this notebook's Sharpe/drawdown to
  Phase 3/4's in-sample numbers, the honest OOS numbers are typically *worse* — that gap *is*
  the lookahead bias Phase 3/4 was quietly benefiting from. Showing both, side by side, is a
  stronger submission than only showing the good-looking number.
- **Why `MIN_DWELL`.** Raw daily HMM output chatters between states even when the underlying
  process is a genuinely persistent regime — document the run-length numbers this notebook
  printed in Section 7 as your justification, and note the filter is strictly backward-looking.
- **Why `MAX_WEIGHT = 0.65`.** With only 3 assets and a long-only, sum-to-1 constraint,
  unconstrained mean-variance optimization tends to produce corner solutions (100% in one
  asset). The cap is a standard, still-convex way to keep the optimizer honest without hiding
  the direction it wants to go.
- **Fallback logic.** Early folds may not have seen a Crisis yet during training — in that case
  the optimizer falls back to the full training-period mean/covariance rather than fitting on
  a handful of noisy days. Worth a sentence in the README on how often this fallback triggered.
- **VIX / ticker choice.** Confirm `ASSET_COLS` and any VIX-based feature actually point at the
  Indian VIX (`^INDIAVIX`) and NSE-appropriate instruments, not US analogues — easy to carry
  over by accident from a generic demo notebook.
